<a href="https://colab.research.google.com/github/VitorTardivo21/Redes-Neurais-e-IA-Aplicada/blob/main/01_exploracao_site.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 20.9 MB/s eta 0:00:00


In [ ]:
# Instalar uma única vez no Colab
# !pip install pypdf

import requests
import pandas as pd
import json
import base64

from datetime import datetime
from pypdf import PdfReader
from io import BytesIO

# ==========================================
# CONFIGURAÇÃO
# ==========================================

DATA_INICIAL = "2026-05-25"
DATA_FINAL   = "2026-05-31"

ARQUIVO_SAIDA = "diarios_avare.csv"

# ==========================================
# API AVARÉ
# ==========================================

API_URL = "https://dosp.com.br/api/index.php/dioe.js/4700?callback=dioe"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

print("Baixando lista de edições...")

r = requests.get(API_URL, headers=HEADERS)

if r.status_code != 200:
    raise Exception(f"Erro ao acessar API ({r.status_code})")

# Remove callback JSONP
texto = r.text
json_text = texto[texto.find("(")+1:texto.rfind(")")]

dados_api = json.loads(json_text)

print(f"Total de edições disponíveis: {len(dados_api['data'])}")

# ==========================================
# FILTRO DE DATAS
# ==========================================

data_inicial = datetime.strptime(DATA_INICIAL, "%Y-%m-%d")
data_final = datetime.strptime(DATA_FINAL, "%Y-%m-%d")

edicoes = []

for item in dados_api["data"]:

    try:

        data_edicao = datetime.strptime(
            item["data"][:10],
            "%Y-%m-%d"
        )

        if data_inicial <= data_edicao <= data_final:
            edicoes.append(item)

    except:
        pass

print(f"Edições encontradas no período: {len(edicoes)}")

# ==========================================
# DOWNLOAD DOS PDFS
# ==========================================

dados = []

for posicao, item in enumerate(edicoes, start=1):

    try:

        iddo = item["iddo"]

        codigo = base64.b64encode(
            str(iddo).encode()
        ).decode()

        url_pdf = f"https://dosp.com.br/exibe_do.php?i={codigo}"

        print(
            f"[{posicao}/{len(edicoes)}] "
            f"Edição {item['edicao_do']} "
            f"({item['data'][:10]})"
        )

        resposta_pdf = requests.get(
            url_pdf,
            headers=HEADERS,
            timeout=60
        )

        if resposta_pdf.status_code != 200:
            print("Erro ao baixar PDF")
            continue

        reader = PdfReader(
            BytesIO(resposta_pdf.content)
        )

        texto_documento = ""

        for pagina in reader.pages:

            try:
                texto_documento += (
                    pagina.extract_text() or ""
                )
                texto_documento += "\n"
            except:
                pass

        dados.append({
            "data": item["data"][:10],
            "edicao": item["edicao_do"],
            "paginas": item.get("pgtotal"),
            "url": url_pdf,
            "conteudo": texto_documento
        })

    except Exception as e:

        print(
            f"Erro na edição "
            f"{item.get('edicao_do')} -> {e}"
        )

# ==========================================
# SALVAR CSV
# ==========================================

df = pd.DataFrame(dados)

df.to_csv(
    ARQUIVO_SAIDA,
    index=False,
    encoding="utf-8-sig"
)

print("\n=================================")
print("FINALIZADO")
print("Registros salvos:", len(df))
print("Arquivo:", ARQUIVO_SAIDA)
print("=================================")

print(df.head())

Baixando lista de edições...
Total de edições disponíveis: 2535
Edições encontradas no período: 7
[1/7] Edição 2746 (2026-05-29)
[2/7] Edição 2745 (2026-05-29)
[3/7] Edição 2744 (2026-05-28)
[4/7] Edição 2743 (2026-05-28)
[5/7] Edição 2742 (2026-05-27)
[6/7] Edição 2741 (2026-05-26)
[7/7] Edição 2740 (2026-05-25)

FINALIZADO
Registros salvos: 7
Arquivo: diarios_avare.csv
         data edicao  paginas                                          url  \
0  2026-05-29   2746       18  https://dosp.com.br/exibe_do.php?i=ODI1NzU3   
1  2026-05-29   2745       61  https://dosp.com.br/exibe_do.php?i=ODI1MzUz   
2  2026-05-28   2744        4  https://dosp.com.br/exibe_do.php?i=ODI1MTI1   
3  2026-05-28   2743        8  https://dosp.com.br/exibe_do.php?i=ODI0NzI0   
4  2026-05-27   2742       27  https://dosp.com.br/exibe_do.php?i=ODI0Mjc5   

                                            conteudo  
0  SEMANÁRIO\nOﬁcial Eletrônico\navare.sp.gov.br\...  
1  SEMANÁRIO\nOﬁcial Eletrônico\navare.sp.gov.b

In [ ]:
import requests
from pypdf import PdfReader
from io import BytesIO

url = "https://dosp.com.br/exibe_do.php?i=ODI1NzU3"

pdf_bytes = requests.get(url).content

reader = PdfReader(BytesIO(pdf_bytes))

texto = ""

for pagina in reader.pages:
    texto += pagina.extract_text() or ""

print(texto[:3000])

SEMANÁRIO
Oﬁcial Eletrônico
avare.sp.gov.br
Sexta-feira, 29 de maio de 2026 Ano X | Edição nº 2746 Prefeito: Roberto Araujo
PODER EXECUTIVO
Poder Executivo
Atos Oﬁciais
Atos Oﬁciais
Decretos
Decretos
Decreto n.º 8.758, de 26 de maio de 2026
(Reorganiza o Conselho Municipal
dos  Direitos  da  Criança  e  do
Adolescente - CMDCA).
ROBERTO DE ARAUJO , Prefeito da Estância Turística
de  Avaré,  usando  das  suas  atribuições  que  lhe  são
conferidas por lei,
DECRETA:
Artigo 1º  – Fica reorganizado o Conselho Municipal
dos Direitos da Criança e do Adolescente, para o biênio
2025-2027, na forma abaixo, em conformidade com a Lei
Federal n.º 8.069/90, a Lei Complementar n.º 150/2011
(Art. 28, § 2º), Regimento Interno (Art. 16 e 18) e decisão
da Reunião Plenária do CMDCA, realizada em 05 de maio de
2026:
Presidente: Tatyane de Paula Montagno Pereira
Vice-presidente: Caroline da Silva Lopes
Secretário: Clovis Rodrigues Felipe
I – Representantes do Poder Público:
1. Secretaria Municipal de Educaç